# AI Programming — Lecture 21
## Lab 1: CLIP & SigLIP Zero-Shot Classification

이번 실습에서는 **사전학습된 CLIP과 SigLIP**을 이용하여
이미지를 추가 학습 없이 분류합니다.

### 학습 목표

- Pretrained VLM을 불러와 inference할 수 있습니다.
- 이미지와 텍스트를 같은 embedding 공간에서 비교하는 방식을 이해합니다.
- CLIP의 **zero-shot classification**을 직접 수행합니다.
- Prompt가 prediction score에 영향을 줄 수 있음을 확인합니다.
- 같은 이미지와 class 후보에 대해 CLIP과 SigLIP의 결과를 비교합니다.

### 핵심 아이디어

```text
Image
  ↓
Image Encoder
  ↓
Image Embedding
      ↕ similarity
Text Embedding
  ↑
Text Encoder
  ↑
Candidate Labels
```

> 이번 실습에서는 model을 fine-tuning하지 않습니다.  
> 이미 학습된 image-text representation을 그대로 사용합니다.

## 1. 필요한 패키지 설치

CLIP은 OpenAI의 공개 repository에서 설치하고,
SigLIP은 Hugging Face Transformers를 사용합니다.

In [ ]:
!pip install -q "git+https://github.com/openai/CLIP.git" pillow
!pip install -q -U transformers accelerate

## 2. 이미지 불러오기

In [ ]:
import torch
import clip

from PIL import Image
import requests
from io import BytesIO
from IPython.display import display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# 필요하면 URL만 바꾸어 사용하세요.
image_url = "https://images.pexels.com/photos/617278/pexels-photo-617278.jpeg"

response = requests.get(image_url)
response.raise_for_status()

image = Image.open(
    BytesIO(response.content)
).convert("RGB")

display(image)

## 3. CLIP Zero-Shot Classification

원래 사용했던 CLIP 실습과 같은 방식입니다.

### Candidate labels

```text
a cat
a dog
a car
a person
a tree
```

각 label을 text encoder에 넣어 embedding을 만들고,
image embedding과 cosine similarity를 비교합니다.

In [ ]:
# 1) CLIP 모델 로드
clip_model, clip_preprocess = clip.load(
    "ViT-B/32",
    device=device
)

clip_model.eval()

# 2) 분류 후보
class_names = [
    "a cat",
    "a dog",
    "a car",
    "a person",
    "a tree",
]

print("클래스 라벨:", class_names)

# 3) Text tokenization
text_tokens = clip.tokenize(
    class_names
).to(device)

# 4) Image / Text embedding
with torch.no_grad():
    image_input = (
        clip_preprocess(image)
        .unsqueeze(0)
        .to(device)
    )

    image_features = clip_model.encode_image(
        image_input
    )

    text_features = clip_model.encode_text(
        text_tokens
    )

    # L2 normalization
    image_features /= image_features.norm(
        dim=-1,
        keepdim=True
    )

    text_features /= text_features.norm(
        dim=-1,
        keepdim=True
    )

    # cosine similarity
    logits_per_image = (
        image_features
        @ text_features.T
    )

    probs = (
        logits_per_image
        .softmax(dim=-1)
        .cpu()
        .numpy()[0]
    )

print("\n=== CLIP 분류 결과 ===")

for cls, p in sorted(
    zip(class_names, probs),
    key=lambda x: x[1],
    reverse=True,
):
    print(
        f"{cls:15s}: {p*100:5.2f}%"
    )

### 확인할 내용

CLIP은 별도의 CIFAR/ImageNet classifier를 학습하지 않고도
**텍스트 자체를 class description으로 사용**할 수 있습니다.

이것이 zero-shot classification의 핵심입니다.

## 4. Prompt Engineering

같은 class라도 prompt를 조금 다르게 표현하면
text embedding이 달라지고 similarity score도 달라질 수 있습니다.

아래에서는 `cat` class를 세 가지 방식으로 표현해 봅니다.

In [ ]:
prompt_sets = {
    "short": [
        "cat",
        "dog",
        "car",
        "person",
        "tree",
    ],
    "article": [
        "a cat",
        "a dog",
        "a car",
        "a person",
        "a tree",
    ],
    "photo": [
        "a photo of a cat",
        "a photo of a dog",
        "a photo of a car",
        "a photo of a person",
        "a photo of a tree",
    ],
}

with torch.no_grad():
    for name, prompts in prompt_sets.items():
        text_tokens = clip.tokenize(
            prompts
        ).to(device)

        text_features = clip_model.encode_text(
            text_tokens
        )

        text_features /= text_features.norm(
            dim=-1,
            keepdim=True
        )

        logits = (
            image_features
            @ text_features.T
        )

        probs = (
            logits
            .softmax(dim=-1)
            .cpu()
            .numpy()[0]
        )

        best_idx = probs.argmax()

        print(
            f"{name:8s} → "
            f"{prompts[best_idx]:20s} "
            f"({probs[best_idx]*100:5.2f}%)"
        )

## 5. SigLIP Zero-Shot Classification

SigLIP도 image encoder와 text encoder를 사용하는 dual-encoder VLM입니다.

CLIP과 가장 큰 차이는 **pretraining loss**입니다.

```text
CLIP
→ softmax-based contrastive loss

SigLIP
→ pairwise sigmoid loss
```

Inference에서는 같은 이미지와 text 후보를 넣어
image-text matching score를 비교할 수 있습니다.

In [ ]:
from transformers import (
    AutoModel,
    AutoProcessor,
)

siglip_model_name = (
    "google/siglip-base-patch16-224"
)

siglip_processor = (
    AutoProcessor.from_pretrained(
        siglip_model_name
    )
)

siglip_model = (
    AutoModel.from_pretrained(
        siglip_model_name
    )
    .to(device)
)

siglip_model.eval()

siglip_labels = [
    "cat",
    "dog",
    "car",
    "person",
    "tree",
]

# SigLIP에서 권장되는 형태의 prompt
siglip_prompts = [
    f"This is a photo of {label}."
    for label in siglip_labels
]

inputs = siglip_processor(
    text=siglip_prompts,
    images=image,
    padding="max_length",
    return_tensors="pt",
).to(device)

with torch.no_grad():
    outputs = siglip_model(**inputs)

    # SigLIP은 pairwise sigmoid score 사용
    siglip_probs = (
        torch.sigmoid(
            outputs.logits_per_image
        )
        .cpu()
        .numpy()[0]
    )

print("\n=== SigLIP 분류 결과 ===")

for cls, p in sorted(
    zip(siglip_labels, siglip_probs),
    key=lambda x: x[1],
    reverse=True,
):
    print(
        f"{cls:15s}: {p*100:5.2f}%"
    )

### CLIP과 SigLIP score를 볼 때 주의할 점

CLIP 예제에서는 여러 candidate label에 softmax를 적용했기 때문에
score의 합이 1이 됩니다.

SigLIP은 각 image-text pair에 sigmoid를 적용하므로
각 score가 **독립적인 matching score**이며 합이 1일 필요가 없습니다.

따라서 두 모델에서는 **순위와 상대적인 matching 경향**을 비교하는 것이 좋습니다.

## 6. 직접 해보기

1. `image_url`을 다른 이미지로 바꾸어 보세요.
2. `class_names`에 새로운 후보를 추가해 보세요.
3. 다음 prompt를 비교해 보세요.

```text
dog
a dog
a photo of a dog
a photo of a dog, a type of pet
```

4. CLIP과 SigLIP의 prediction ranking이 같은지 비교하세요.
5. 후보 label에 정답과 매우 비슷한 표현을 여러 개 넣으면 score가 어떻게 변하는지 확인하세요.

# 정리

```text
CLIP / SigLIP
Image ↔ Text Alignment
        ↓
Zero-Shot Classification
```

### 꼭 기억할 것

1. Pretrained VLM은 새로운 classifier를 학습하지 않고도 text prompt를 label로 사용할 수 있습니다.
2. CLIP은 image와 text를 shared embedding space에서 비교합니다.
3. Prompt 표현에 따라 text embedding과 prediction이 달라질 수 있습니다.
4. SigLIP은 기본 dual-encoder 구조는 유지하면서 sigmoid-based pretraining objective를 사용합니다.